# DICE — Notebook 1.2 (Scenarios & Forcing) — Solutions



**Learning objectives (1h30min)**
0. Load a minimal DICE code and identify its **blocks**: parameters, initial states, economic equations, climate equations.  
1. SSP Scenarios: generate meaningful SSP scenarios by adjusting the calibration of exogenous variables.
2. Introduce some forcing objectives, by adjusting

> This notebook uses the provided `DICE.py` module (teaching version).


# 0) Load DICE and run baseline


In [ ]:
# 1) Load DICE and baseline run
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)


p = Params()
path = init_states(p)
path[:, p.i_mu] = 0.03          # baseline abatement path
path = update_path(path, range(1, p.nT), p)
df_base = mat_to_df(path, p)
df_base.head()


## 1) SSP Scenarios

In what follows, we want to simulate two extreme narratives, the **SSP1** and **SSP5** storylines, as well as the **baseline (SSP2)**.  

- **SSP1** – *Sustainability: Taking the Green Road (Low challenges to mitigation and adaptation)*  
  *The world shifts gradually toward a more sustainable path, emphasizing inclusive development and respect for environmental boundaries. Educational and health investments accelerate the demographic transition, inequality is reduced, and consumption patterns shift toward low material growth and lower resource and energy intensity.*

- **SSP5** – *Fossil-fueled Development: Taking the Highway (High challenges to mitigation, low challenges to adaptation)*  
  *This world emphasizes competitive markets, innovation, and human capital development. Rapid economic growth is fueled by fossil resources and energy-intensive lifestyles. Local environmental problems are managed, but global commons are stressed, with high emissions and reliance on future technological fixes or geo-engineering.*

We thus consider three scenarios: the **optimistic** (SSP1), the **baseline** (SSP2, current calibration), and the **pessimistic** (SSP5).  
The table below summarizes the proposed core calibration changes:

| Parameter (description)                | Symbol   | SSP1 (Optimistic)                         | SSP2 (Baseline — DICE) | SSP5 (Pessimistic)                        |
|----------------------------------------|----------|--------------------------------------------|-------------------------|-------------------------------------------|
| Emissions intensity (initial)          | $\sigma_0$ | 25% lower than baseline                     | calibrated value        | 15% higher than baseline                   |
| Decline rate of emissions intensity    | $g_{\sigma}$ | 25% faster decline                         | calibrated value        | 50% slower decline                        |
| Long-run population (asymptote)        | $L_{\infty}$ | 9 billion                                  | calibrated value        | 16 billion                                |
| Population growth parameter            | $l_g$      | 20% lower                           | calibrated value        | 30% higher                        |
| Total factor productivity (TFP) level  | $A_0$      | 5% higher than baseline                    | calibrated value        | 3% lower than baseline                    |
| TFP growth rate                        | $g_A$      | 15% faster than baseline                   | calibrated value        | 20% slower than baseline                  |

> ✍️ **Instructions**: For each scenario, initialize parameters accordingly, run the model, and compare trajectories for output, emissions, forcing, and temperatures.


#### 1-A) Build the calibration and check it works in new p vector.
Use the table to fill each scenarios and have the right narrative.


In [ ]:
# Solution

def make_params_variant(**kwargs):
    q = Params()
    for k,v in kwargs.items():
        setattr(q, k, v)
    return q

p_base = Params()
p_opt  = make_params_variant( sigma0=p_base.sigma0*0.85, gsig=p_base.gsig*(1+0.5),lg=p_base.lg*(1-0.2), Linf=9000,  A0=p_base.A0*1.05, gA=p_base.gA*1.15)
p_pes  = make_params_variant( sigma0=p_base.sigma0*1.25, gsig=p_base.gsig*(1-0.40),lg=p_base.lg*(1+0.3), Linf=16000, A0=p_base.A0*0.95, gA=p_base.gA*0.80)


#### 1-B) Provide some graph comparing Emissions, GDP, damages (define them manually) and temperatures.


In [ ]:
timevec = range(1, p.nT)

# Base
sim_base = init_states(p_base)
sim_base[1:, p_base.i_mu] = 0.03
sim_base[1:,  p_base.i_s]  = 0.20
sim_base = update_path(sim_base, timevec, p_base)
years = sim_base[:, p.i_time]

# Opt
sim_opt = init_states(p_opt)
sim_opt[1:, p_opt.i_mu] = 0.03
sim_opt[1:,  p_opt.i_s]  = 0.20
sim_opt = update_path(sim_opt, timevec, p_opt)

# Opt
sim_pes = init_states(p_pes)
sim_pes[1:, p_pes.i_mu] = 0.03
sim_pes[1:,  p_pes.i_s]  = 0.20
sim_pes = update_path(sim_pes, timevec, p_pes)


plt.figure()
plt.plot(years, sim_base [:,  p_base.i_T_AT], linewidth=2, label="SSP2")
plt.plot(years, sim_opt [:,  p_opt.i_T_AT], linewidth=2, label="SSP1")
plt.plot(years, sim_pes[:,  p_pes.i_T_AT], linewidth=2, label="SSP5")
plt.xlabel("Year"); plt.ylabel("°C")
plt.title("Temperatures")
plt.grid(True); plt.legend()
plt.show()



idx_2100 = int(np.argmin(np.abs(years - 2100)))
print({name: round(sim[idx_2100, par.i_T_AT], 3) for name, sim, par in [
    ("SSP1-like", sim_opt, p_opt), ("SSP2-like", sim_base, p_base), ("SSP5-like", sim_pes, p_pes)
]})


#### 1-C) According to all the figures, do you think that SSPs narrative are the main drivers of climate change? What is the temperatures in best versus worst case of climate change?


> **Interpretation.** The SSP narratives alter population, productivity and carbon intensity, and therefore emissions and temperature. In this simplified DICE calibration they are conditional drivers, not probabilities or forecasts. Read the 2100 values from the simulated paths and report the units and parameter assumptions.


## 2) Mitigation schedules: ramp abatement to reach (near) net-zero by 2100 / 2080 / 2060

Up to now, our scenarios were purely **exogenous**: we varied productivity, population, and emissions intensity to mimic different SSP narratives.  
We now enrich the analysis by introducing a **second decision variable**: the **abatement rate** $\mu_t$, which measures the fraction of industrial emissions avoided through mitigation policies and technologies.  

- In the **baseline** exercises, $\mu_t$ was fixed at a constant, low level (e.g. 3%).  
- In this exercise, we explicitly feed an **abatement trajectory** $\{\mu_t\}$ into the model.  
- Conceptually, pushing $\mu_t \to 1$ means that almost all industrial CO₂ emissions are eliminated. Note, however, that **land-use emissions** $E_{\text{land},t}$ remain exogenous and may stay positive for some time.

This mirrors the way scenarios are constructed in **CMIP6 (Coupled Model Intercomparison Project Phase 6)**, where each SSP storyline can be combined with different **climate policy assumptions**:  
- A **high-ambition mitigation pathway** ramps abatement quickly, reaching **net-zero emissions** by mid-century (e.g. 2060).  
- A **delayed mitigation pathway** reaches net-zero much later (e.g. 2100).  
- The resulting **forcing levels by 2100** (e.g. 2.6, 4.5, 8.5 W/m²) define the well-known **SSP1-2.6**, **SSP2-4.5**, and **SSP5-8.5** scenarios.  


3. Interpret the results in light of the IPCC’s SSP scenario framework: *early and ambitious decarbonization (SSP1-2.6) vs. delayed action with high warming (SSP5-8.5).*  


#### 2-a) Construct linear ramps for $\mu_t$ that reach **net zero** by 2100, 2080, and 2060. 
Recall that in this model, we only have two decision variables in here. Saving remains unchanged $\{s_t\}_{t=t_{0}}^{T}=0.2$ while mitigation path must be adjusted: $\{\mu_t\}_{t=t_{0}}^{T}$. It should gradually increase to 1 to reach net zero at some specific date.
Recall that here, our simulations, including decision variables, are all stacked into matrix $w = [y,x,z]$. In what follows, we modify directly $w$.



In [ ]:
# Build date-based linear abatement ramps.
years = sim_base[:, p_base.i_time]

def abatement_ramp(years, target_year):
    ramp = np.zeros(len(years))
    target = int(np.argmin(np.abs(years - target_year)))
    ramp[1:target + 1] = np.linspace(0.0, 1.0, target)
    ramp[target + 1:] = 1.0
    return ramp

ramp2060 = abatement_ramp(years, 2060)
ramp2080 = abatement_ramp(years, 2080)
ramp2100 = abatement_ramp(years, 2100)

plt.figure()
plt.plot(years, ramp2100, linewidth=2, label="2100")
plt.plot(years, ramp2080, linewidth=2, label="2080")
plt.plot(years, ramp2060, linewidth=2, label="2060")
plt.xlabel("Year"); plt.ylabel("Abatement share")
plt.title("Abatement ramps"); plt.grid(True); plt.legend(); plt.show()


#### 2-b) Simulate the model under each ramp and compare the paths for **forcing $F_t$**, **temperatures $T_{AT,t}$**, and **emissions $E_t$**.
Recall that once the matrix of decision $z$ of size $I\times 2$ has been modified (as part of matri $w$), we can once again solve the new system with alternative climate policy:
  $$
  y_t = f_p (y_{t-1}, x_{t-1}, z_t), \quad f: \mathbb{R}^{N_y}\times \mathbb{R}^{N_x}\times \mathbb{R}^2 \to \mathbb{R}^{N_y}.
  $$

  How effective is climate policy in curbing emissions?


In [ ]:
# Simulating 2100 NZ
sim2100  = sim_base.copy()
sim2100[:, p_base.i_mu] =  np.asarray(ramp2100).squeeze()
sim2100 = update_path(sim2100, timevec, p_base)

# Simulating 2080 NZ
sim2080  = sim_base.copy()
sim2080[:, p_base.i_mu] =  np.asarray(ramp2080).squeeze()
sim2080 = update_path(sim2080, timevec, p_base)

# Simulating 2060 NZ
sim2060  = sim_base.copy()
sim2060[:, p_base.i_mu] =  np.asarray(ramp2060).squeeze()
sim2060 = update_path(sim2060, timevec, p_base)



fig, axes = plt.subplots(1, 3, figsize=(15, 4))  # wider figure

# --- Forcing
axes[0].plot(years, sim2100[:, p_base.i_F], linewidth=2, label="2100")
axes[0].plot(years, sim2080[:, p_base.i_F], linewidth=2, label="2080")
axes[0].plot(years, sim2060[:, p_base.i_F], linewidth=2, label="2060")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("W/m²")
axes[0].set_title("Forcing"); axes[0].grid(True); axes[0].legend()

# --- Temperature
axes[1].plot(years, sim2100[:, p_base.i_T_AT], linewidth=2, label="2100")
axes[1].plot(years, sim2080[:, p_base.i_T_AT], linewidth=2, label="2080")
axes[1].plot(years, sim2060[:, p_base.i_T_AT], linewidth=2, label="2060")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("°C")
axes[1].set_title("T_AT"); axes[1].grid(True); axes[1].legend()

# --- Emissions
axes[2].plot(years, sim2100[:, p_base.i_E], linewidth=2, label="2100")
axes[2].plot(years, sim2080[:, p_base.i_E], linewidth=2, label="2080")
axes[2].plot(years, sim2060[:, p_base.i_E], linewidth=2, label="2060")
axes[2].set_xlabel("Year"); axes[2].set_ylabel("GtC")
axes[2].set_title("E"); axes[2].grid(True); axes[2].legend()

# Adjust spacing
fig.tight_layout(w_pad=3.0)  # add horizontal padding
plt.show()

print({name: round(sim[idx_2100, p_base.i_T_AT], 3) for name, sim in [
    ("NZ2060", sim2060), ("NZ2080", sim2080), ("NZ2100", sim2100)
]})


> **Interpretation.** Earlier net zero lowers cumulative emissions, forcing and end-of-century temperature more strongly. The ordering is conditional on the common saving rule and on DICE's carbon-cycle and temperature equations.


### 2-c) What is the economic cost of climate mitigation? 
Provide a 2 plots, one showing the consumption gap defined as $100\times (c_t^{alt}/c_t^{base}-1)$ and in a second figure the temperature gap  $(T_t^{alt}-T_t^{base})$. $T_t^{alt}$ denotes the temperature path in alternative regimes (e.g. optimistic or pessimistic) while *base* is baseline.  
Comment the result your find about the sacrifice that must be done by society in terms of consumption loss during the transition. What drives this sacrifice?


In [ ]:
# Solution
gapC_2060 = 100*(sim2060[:, p_base.i_C] - sim_base[:, p_base.i_C])/sim_base[:, p_base.i_C]
gapC_2080 = 100*(sim2080[:, p_base.i_C] - sim_base[:, p_base.i_C])/sim_base[:, p_base.i_C]
gapC_2100 = 100*(sim2100[:, p_base.i_C] - sim_base[:, p_base.i_C])/sim_base[:, p_base.i_C]


gapT_2060 = sim2060[:, p_base.i_T_AT] - sim_base[:, p_base.i_T_AT]
gapT_2080 = sim2080[:, p_base.i_T_AT] - sim_base[:, p_base.i_T_AT]
gapT_2100 = sim2100[:, p_base.i_T_AT] - sim_base[:, p_base.i_T_AT]


# --- plot ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (1) Consumption gap (%)
ax = axes[0]
ax.plot(years, gapC_2100, linewidth=2, label="NZ by 2100")
ax.plot(years, gapC_2080, linewidth=2, label="NZ by 2080")
ax.plot(years, gapC_2060, linewidth=2, label="NZ by 2060")
ax.axhline(0, color="k", linewidth=1, alpha=0.4)
ax.set_title("Consumption gap vs. baseline")
ax.set_xlabel("Year"); ax.set_ylabel("% (alt / base − 1) × 100")
ax.grid(True); ax.legend()

# (2) Temperature gap (°C)
ax = axes[1]
ax.plot(years, gapT_2100, linewidth=2, label="NZ by 2100")
ax.plot(years, gapT_2080, linewidth=2, label="NZ by 2080")
ax.plot(years, gapT_2060, linewidth=2, label="NZ by 2060")
ax.axhline(0, color="k", linewidth=1, alpha=0.4)
ax.set_title("Temperature gap vs. baseline")
ax.set_xlabel("Year"); ax.set_ylabel("°C")
ax.grid(True); ax.legend()

fig.tight_layout(w_pad=3.0)
plt.show()
print("2100 consumption gaps (%):", {"NZ2060": round(gapC_2060[idx_2100], 3),
      "NZ2080": round(gapC_2080[idx_2100], 3), "NZ2100": round(gapC_2100[idx_2100], 3)})


> **Interpretation.** Faster mitigation reduces consumption initially through abatement costs, while lowering future warming. The comparison is not a welfare ranking: it omits deep uncertainty, distributional effects and model error.


## 3) Providing core scenarios mixing SSPs $\times$ Forcing as in IPCC


The table below shows how **SSP storylines** combine with **radiative forcing targets** (W/m² by 2100).  
For example, *SSP1-2.6* corresponds to the **SSP1 narrative** combined with a **2.6 W/m² forcing pathway**.

|      | **1.9** | **2.6** | **4.5** | **7.0** | **8.5** |
|------|---------|---------|---------|---------|---------|
| **SSP1** | SSP1-1.9 | SSP1-2.6 | –       | –       | –       |
| **SSP2** | –       | SSP2-2.6 | SSP2-4.5 | SSP2-7.0 | –       |
| **SSP3** | –       | –       | SSP3-4.5 | SSP3-7.0 | SSP3-8.5 |
| **SSP4** | –       | SSP4-2.6 | SSP4-4.5 | SSP4-7.0 | –       |
| **SSP5** | –       | –       | SSP5-4.5 | –       | SSP5-8.5 |

### Approximate Mapping: Forcing → Global Warming by 2100

Radiative forcing pathways translate into different **global mean temperature increases** relative to preindustrial levels (1850–1900).  
Values below are **approximate central estimates** (IPCC AR6, median climate sensitivity assumptions):

- **1.9 W/m² (SSP1-1.9)** → ~ **1.5 °C** stabilization (consistent with Paris Agreement “1.5 °C” target).  
- **2.6 W/m² (SSP1/2/4-2.6)** → ~ **2.0 °C** warming by 2100 (low-forcing pathway).  
- **4.5 W/m² (SSP2/3/4/5-4.5)** → ~ **2.5–3.0 °C** warming by 2100 (intermediate pathway).  
- **7.0 W/m² (SSP2/3/4-7.0)** → ~ **3.5–4.0 °C** warming by 2100 (high-forcing pathway).  
- **8.5 W/m² (SSP3/5-8.5)** → ~ **4.5 °C or more** warming by 2100 (very high emissions, “worst-case” reference).

---

👉 **Interpretation**:  
- Lower forcing pathways (SSP1-1.9, SSP1/2-2.6) require **early, ambitious mitigation** and typically correspond to net-zero CO₂ by mid-century.  
- Higher forcing pathways (SSP3-7.0, SSP5-8.5) assume **limited mitigation**, continued fossil fuel use, and delayed or absent net-zero transitions.  
- Intermediate cases (SSP2-4.5) represent **current policy trajectories** or delayed mitigation efforts.  


#### 3-A) Some core scenarii outside the diagonal are not evaluated (e.g. SSP1-8.5). Why?


> **Answer.** SSP and forcing labels are combined only when the socioeconomic narrative, emissions pathway and radiative-forcing outcome are internally coherent. An off-diagonal experiment can still be simulated as a stress test, but it must not be presented as an assessed IPCC pathway or a forecast.


#### 3-B) Try to compute the SSP1-1.9, which corresponds to net zero by 2060 and optimistic scenario. Is SSP1-1.9 out of reach?


In [ ]:
# 1) Build a fresh path using p_opt exogenous (safer than copying sim_base)
sim2060_opt = init_states(p_opt)

# Reuse the same saving policy as baseline (or choose your own)
sim2060_opt[:, p_opt.i_s] = sim_base[:, p_base.i_s]

# Impose NZ-by-2060 abatement path
sim2060_opt[:, p_opt.i_mu] = np.asarray(ramp2060).squeeze()

# 2) Simulate once (no need to call update_path twice)
sim2060_opt = update_path(sim2060_opt, timevec, p_opt)

# 3) Plot Forcing, Temperature, Emissions vs baseline
years = sim_base[:, p_base.i_time]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Forcing
axes[0].plot(years, sim_base[:,    p_base.i_F],     lw=2, label="Baseline")
axes[0].plot(years, sim2060_opt[:, p_opt.i_F],      lw=2, label="SSP1–1.9 (NZ 2060, opt)")
axes[0].set_title("Forcing")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("W/m²")
axes[0].grid(True); axes[0].legend()

# --- Temperature
axes[1].plot(years, sim_base[:,    p_base.i_T_AT],  lw=2, label="Baseline")
axes[1].plot(years, sim2060_opt[:, p_opt.i_T_AT],   lw=2, label="SSP1–1.9")
axes[1].set_title("T_AT")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("°C")
axes[1].grid(True); axes[1].legend()

# --- Emissions
axes[2].plot(years, sim_base[:,    p_base.i_E],     lw=2, label="Baseline")
axes[2].plot(years, sim2060_opt[:, p_opt.i_E],      lw=2, label="SSP1–1.9")
axes[2].set_title("Emissions")
axes[2].set_xlabel("Year"); axes[2].set_ylabel("GtC")
axes[2].grid(True); axes[2].legend()

fig.tight_layout(w_pad=3.0)
plt.show()

print("SSP1-like + NZ2060 temperature in 2100:", round(sim2060_opt[idx_2100, p_opt.i_T_AT], 3), "°C")


> **Answer.** In this simplified model, SSP1–1.9 is represented by an optimistic socioeconomic calibration plus rapid abatement. Feasibility here means numerical attainability under imposed controls, not political or technological likelihood.


#### 3-C) Is it possible to obtain a 4°C of warming in SSP1? Evaluate SSP1 under no mitigation policy with $\mu=0$ to mimic SSP1-8.5


In [ ]:
# SSP1-like socioeconomic assumptions under no mitigation.
sim_no_mitigation_opt = init_states(p_opt)
sim_no_mitigation_opt[:, p_opt.i_s] = sim_base[:, p_base.i_s]
sim_no_mitigation_opt[:, p_opt.i_mu] = 0.0
sim_no_mitigation_opt = update_path(sim_no_mitigation_opt, timevec, p_opt)

years = sim_no_mitigation_opt[:, p_opt.i_time]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column, title, unit in zip(
    axes,
    [p_opt.i_F, p_opt.i_T_AT, p_opt.i_E],
    ["Forcing", "Temperature", "Emissions"],
    ["W/m²", "°C", "GtCO₂ per period"],
):
    ax.plot(years, sim_base[:, column], lw=2, label="Baseline")
    ax.plot(years, sim_no_mitigation_opt[:, column], lw=2,
            label="SSP1-like / no mitigation")
    ax.set_title(title); ax.set_xlabel("Year"); ax.set_ylabel(unit)
    ax.grid(True); ax.legend()
fig.tight_layout(); plt.show()

idx_2100 = int(np.argmin(np.abs(years - 2100)))
print("SSP1-like / no-mitigation warming in 2100:",
      round(float(sim_no_mitigation_opt[idx_2100, p_opt.i_T_AT]), 2), "°C")


> **Answer.** The no-mitigation stress test shows whether an optimistic socioeconomic calibration can still produce high warming. Its result is conditional on this DICE parameterisation and should not be relabelled as an official SSP1–8.5 pathway.


#### 3-D) Is it possible to obtain a 1.5°C in pessimistic environment? Evaluate SSP5 under mitigation policy with $\mu=1$ by 2060 to mimic SSP5-1.9


In [ ]:
# Solution
# Solution
# 1) Build a fresh path using p_opt exogenous (safer than copying sim_base)
sim2060_pes = init_states(p_pes)

# Reuse the same saving policy as baseline (or choose your own)
sim2060_pes[:, p_pes.i_s] = sim_base[:, p_pes.i_s]

# Impose no abatement path
sim2060_pes[:, p_pes.i_mu] = np.asarray(ramp2060).squeeze()

# 2) Simulate once (no need to call update_path twice)
sim2060_pes = update_path(sim2060_pes, timevec, p_pes)

# 3) Plot Forcing, Temperature, Emissions vs baseline
years       = sim2060_pes[:, p_pes.i_time]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Forcing
axes[0].plot(years, sim_base[:,    p_base.i_F],     lw=2, label="Baseline")
axes[0].plot(years, sim2060_pes[:, p_pes.i_F],      lw=2, label="SSP5-like + NZ2060 stress test")
axes[0].set_title("Forcing")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("W/m²")
axes[0].grid(True); axes[0].legend()

# --- Temperature
axes[1].plot(years, sim_base[:,    p_base.i_T_AT],  lw=2, label="Baseline")
axes[1].plot(years, sim2060_pes[:, p_pes.i_T_AT],   lw=2, label="SSP5-like + NZ2060 stress test")
axes[1].set_title("T_AT")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("°C")
axes[1].grid(True); axes[1].legend()

# --- Emissions
axes[2].plot(years, sim_base[:,    p_base.i_E],     lw=2, label="Baseline")
axes[2].plot(years, sim2060_pes[:, p_pes.i_E],      lw=2, label="SSP5-like + NZ2060 stress test")
axes[2].set_title("Emissions")
axes[2].set_xlabel("Year"); axes[2].set_ylabel("GtC")
axes[2].grid(True); axes[2].legend()

fig.tight_layout(w_pad=3.0)
plt.show()


print("SSP5-like + NZ2060 temperature in 2100:", round(sim2060_pes[idx_2100, p_pes.i_T_AT], 3), "°C")


> **Answer.** Strong abatement can substantially limit warming even under the pessimistic calibration, but reaching exactly 1.5°C is a model output, not an assumed guarantee. Report the simulated endpoint and the residual gap explicitly.
